In [2]:
import pandas as pd

data = pd.read_csv("../data/train-2025.csv", low_memory=False)
len(data)

1023552

In [3]:
# Flags for economically implausible values; see docs/project-update/features/financial-anomalies.md
# Original values are never changed. Run on raw data, before fill_missing (03_missing_values.ipynb), so flags reflect reported values, not imputed ones.
DEBT_DETAIL = ["debt_bank_st", "debt_bank_lt", "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt"]
NEG_BS_COLS = [
    "asst_intang_fixed", "asst_tang_fixed", "asst_fixed_fin", "AR", "liab_lt", "liab_lt_emp",
    *DEBT_DETAIL, "debt_st", "debt_lt",
]


def flag_anomalies(df):
    """Take raw data, return a copy with five 0/1 flag columns added. Missing inputs never trigger a flag."""
    cols = [*NEG_BS_COLS, "rev_operating", "COGS", "exp_financing", "ebitda", "prof_operations", "prof_financing", "goodwill"]
    x = df[cols].apply(pd.to_numeric, errors="coerce")  # blanks, "NA" etc. become NaN; NaN comparisons are False
    df = df.copy()

    df["neg_bs_flag"] = (x[NEG_BS_COLS] < 0).any(axis=1)
    df["neg_pl_flag"] = (x["rev_operating"] < 0) | (x["COGS"] < 0) | (x["exp_financing"] < 0) | (x["ebitda"] < x["prof_operations"])
    # 0 is a real 0 under our hypothesis (class D value adjustments), so only nonzero reported values count
    df["fin_adj_flag"] = x["prof_financing"].notna() & (x["prof_financing"] != 0)
    df["neg_goodwill_flag"] = x["goodwill"] < 0
    # Mutually exclusive with debt_detail_missing: only rows where all six details are present
    st = x["debt_bank_st"] + x["debt_fin_st"] + x["AP_st"] > x["debt_st"] + 1
    lt = x["debt_bank_lt"] + x["debt_fin_lt"] + x["AP_lt"] > x["debt_lt"] + 1
    df["debt_detail_inconsistent"] = x[DEBT_DETAIL].notna().all(axis=1) & (st | lt)

    flags = ["neg_bs_flag", "neg_pl_flag", "fin_adj_flag", "neg_goodwill_flag", "debt_detail_inconsistent"]
    df[flags] = df[flags].astype(int)
    return df

In [5]:
flagged = flag_anomalies(data)

# Check: original columns untouched; flag counts match the doc
pd.testing.assert_frame_equal(flagged[data.columns], data)
assert flagged["fin_adj_flag"].sum() == 43236
assert flagged["neg_goodwill_flag"].sum() == 107638
assert flagged["debt_detail_inconsistent"].sum() == 11620
assert flagged["neg_bs_flag"].sum() == 6662

# Check: missing-value spellings a holdout may use never trigger a flag
raw = data.head(3).astype(str)
raw.loc[raw.index[0], ["goodwill", "prof_financing", "debt_st"]] = ["", "NA", " "]
assert flag_anomalies(raw).iloc[0][["neg_goodwill_flag", "fin_adj_flag"]].sum() == 0

flags = ["neg_bs_flag", "neg_pl_flag", "fin_adj_flag", "neg_goodwill_flag", "debt_detail_inconsistent"]
y = flagged["def_date"].notna()
pd.DataFrame({"n": flagged[flags].sum(), "default_rate": [y[flagged[f] == 1].mean().round(4) for f in flags]})

,n,default_rate
neg_bs_flag,6662,0.0534
neg_pl_flag,10508,0.0660
fin_adj_flag,43236,0.0609
neg_goodwill_flag,107638,0.0887
debt_detail_inconsistent,11620,0.0458
